# 0. Pyroton V2 Step 2 pilot

Select a T4 GPU runtime. Run the numbered cells in order and authorize your Google Drive.
Upload the companion inputs ZIP in cell 2. All run journals go to Pyroton/V2-step2-pilot in My Drive.
After a disconnect, reconnect the same Drive account and rerun the cells. Completed samples and checks are reused.
This notebook generates and verifies data only. Keep every file under heldout out of training.


In [ ]:
# 1. Connect Drive and check for NVIDIA GPU
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

from google.colab import drive, files

subprocess.run(["nvidia-smi", "-L"], check=True)
mount = Path("/content/drive")
my_drive = mount / "MyDrive"
if not my_drive.is_dir():
    drive.mount(str(mount))
if not my_drive.is_dir():
    raise RuntimeError("Authorize Google Drive from the Files sidebar, then rerun this cell.")

run = my_drive / "Pyroton" / "V2-step2-pilot"
run.mkdir(parents=True, exist_ok=True)
local = Path("/content/pyroton-step2-pilot")
local.mkdir(parents=True, exist_ok=True)
print(run)


In [ ]:
# 2. Load the benchmark and training source bundle
bundle_name = "pyroton-v2-step2-pilot-inputs.zip"
bundle = run / bundle_name
if not bundle.exists():
    uploaded = files.upload()
    if bundle_name not in uploaded:
        raise ValueError(f"Upload the file named {bundle_name}.")
    temporary = bundle.with_suffix(".tmp")
    temporary.write_bytes(uploaded[bundle_name])
    temporary.replace(bundle)

with ZipFile(bundle) as archive:
    if archive.testzip() is not None:
        raise ValueError("The uploaded ZIP is damaged.")
    for member in archive.infolist():
        target = (local / member.filename).resolve()
        if not target.is_relative_to(local.resolve()):
            raise ValueError("Unsafe ZIP member path.")
    archive.extractall(local)

bench = local / "benchmarks"
heldout = local / "heldout"
sources = local / "sources"
results = run / "results"
results.mkdir(exist_ok=True)
os.environ["PYROTON_WASI_CACHE"] = str(local / "wasi-cache")
def invoke(script, *arguments):
    return subprocess.run(
        [sys.executable, str(bench / script), *map(str, arguments)], cwd=local, check=True,
    )


In [ ]:
# 3. Install the pinned verifier and download pinned model/runtime
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wasi-python==0.1.5", "wasmtime==50.0.1"], check=True)

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fetch(url, path, expected):
    if path.is_file() and sha(path) == expected:
        return path
    temporary = path.with_name(path.name + ".download")
    with urlopen(url, timeout=120) as response, temporary.open("wb") as stream:
        shutil.copyfileobj(response, stream, length=4 * 1024 * 1024)
    if sha(temporary) != expected:
        raise RuntimeError(f"SHA-256 mismatch for {path.name}.")
    temporary.replace(path)
    return path

root_url = "https://github.com/ggml-org/llama.cpp/releases/download/b12521/"
backend = local / "llama-runtime"
backend.mkdir(exist_ok=True)
runtime_assets = {
    "llama-b12521-bin-ubuntu-cuda-12.8-x64.tar.gz": "02509065d54065aeff9f71fd15ef22c2f1804742f6823e75cfd67d9e6cdff84a",
    "cudart-llama-b12521-bin-ubuntu-cuda-12.8-x64.tar.gz": "ce5352857ebf9119786f73af2cdcd7fca3d0f0cd826bfeee0567bfe23ebdc81a",
}
for name, digest in runtime_assets.items():
    archive_path = fetch(root_url + name, run / name, digest)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(backend, filter="data")
servers = list(backend.rglob("llama-server"))
if len(servers) != 1:
    raise RuntimeError("Expected one llama-server executable.")
server = servers[0]
library_dirs = sorted({str(path.parent) for path in backend.rglob("*.so*")})
os.environ["LD_LIBRARY_PATH"] = ":".join(library_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
subprocess.run([str(server), "--version"], check=True)
model_url = (
    "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/"
    "f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"
)
cached_model = fetch(model_url, run / "stock-qwen15-q4_k_m.gguf",
                    "cc324af070c2ecbfd324a30884d2f951a7ff756aba85cb811a6ec436933bb046")
model = local / cached_model.name
if not model.exists() or sha(model) != sha(cached_model):
    shutil.copyfile(cached_model, model)


In [ ]:
# 4. Build and seal the audited training corpus
audit_files = {
    "mbpp": sources / "mbpp-full.jsonl", "sanitized": sources / "mbpp-sanitized.json",
    "primary": heldout / "scored-holdout.json", "splits_dir": heldout,
    "old_report": heldout / "primefix-benchmark.json",
    "old_comparison": heldout / "pyroton-mbpp-v1-benchmark.json",
    "mbppplus_release": sources / "MbppPlus-v0.2.0.jsonl.gz",
    "mbppplus_audit": heldout / "v2-mbppplus-audit.json",
}
audit_config = local / "audit-config.json"
audit_config.write_text(json.dumps({k: str(v) for k,v in audit_files.items()}, indent=2))
corpus_args = []
for key, value in audit_files.items():
    corpus_args.extend(["--" + key.replace("_", "-"), value])
invoke("build_pilot_corpus.py", *corpus_args, "--output-dir", results)
corpus = results / "v2-pilot-corpus.json"
verified = results / "verified-corpus.json"


In [ ]:
# 5. Run stock at 512 tokens with a same-GPU 256-token control
suites = {"primary": heldout / "scored-holdout.json", "mbppplus": heldout / "v2-mbppplus-test.json"}
for limit in (512, 256):
    for suite, questions in suites.items():
        generations = results / f"stock{limit}-{suite}.jsonl"
        checks = results / f"stock{limit}-{suite}-scores.json"
        invoke("generate_holdout.py", "--server", server, "--model", model, "--dataset", questions,
               "--output", generations, "--gpu-layers", 99, "--tokens", limit)
        invoke("score_holdout.py", "--dataset", questions, "--generations", generations, "--output", checks)

invoke("report_pilot_baseline.py",
       "--primary", suites["primary"], "--dev", heldout / "dev.json", "--mbppplus", suites["mbppplus"],
       "--old-baseline", heldout / "stock256-step1.json",
       "--primary-generations", results / "stock512-primary.jsonl",
       "--primary-scores", results / "stock512-primary-scores.json",
       "--mbppplus-generations", results / "stock512-mbppplus.jsonl",
       "--mbppplus-scores", results / "stock512-mbppplus-scores.json",
       "--matched256-primary-generations", results / "stock256-primary.jsonl",
       "--matched256-primary-scores", results / "stock256-primary-scores.json",
       "--matched256-mbppplus-generations", results / "stock256-mbppplus.jsonl",
       "--matched256-mbppplus-scores", results / "stock256-mbppplus-scores.json",
       "--output", results / "stock512-baseline.json")


In [ ]:
# 6. Validate references and freeze generated edge inputs
preflight = results / "verifier-preflight.json"
saved_report = local / "evidence" / "reference-preflight.json"
saved_corpus = local / "evidence" / "verified-corpus.json"
checks = {
    "corpus_sha256": sha(corpus),
    "verifier_sha256": sha(bench / "pilot_verifier.py"),
    "edge_generator_sha256": sha(bench / "pilot_edges.py"),
}
if not verified.exists() and not preflight.exists() and saved_report.exists() and saved_corpus.exists():
    proof = json.loads(saved_report.read_bytes())
    approved = json.loads(saved_corpus.read_bytes())
    if (proof.get("complete") and all(proof.get(k) == v for k,v in checks.items())
            and [row["id"] for row in approved] == proof["accepted_ids"]):
        for source, target in ((saved_corpus, verified), (saved_report, preflight)):
            temporary = target.with_suffix(".tmp")
            shutil.copyfile(source, temporary)
            temporary.replace(target)
        print(f"Reusing the frozen preflight of {len(approved)} verified problems.")
journal = verified.with_suffix(".progress.jsonl")
if not verified.exists() or not preflight.exists():
    if journal.exists():
        sys.path.insert(0, str(bench))
        from pilot_io import read_jsonl_repair
        if not read_jsonl_repair(journal):
            journal.unlink()
    invoke("pilot_verifier.py", "--corpus", corpus, "--output", verified,
           "--report", preflight)
proof = json.loads(preflight.read_bytes())
approved = json.loads(verified.read_bytes())
if (not proof.get("complete") or any(proof.get(k) != v for k,v in checks.items())
        or [row["id"] for row in approved] != proof["accepted_ids"]):
    raise ValueError("The saved reference preflight does not match this audited corpus and verifier.")


In [ ]:
# 7. Sample eight completions per verified problem
samples = results / "samples.jsonl"
invoke("sample_pilot.py", "--server", server, "--model", model,
       "--verified-corpus", verified, "--output", samples,
       "--gpu-layers", 99, "--slots", 12, "--threads", 2)


In [ ]:
# 8. Score candidates in sandboxed subprocesses
judgments = results / "judgments.jsonl"
invoke("judge_pilot.py", "--verified-corpus", verified, "--samples", samples, "--output", judgments)


In [ ]:
# 9. Repeat the audit and export the two data files
invoke("report_pilot.py", "--verified-corpus", verified, "--judgments", judgments,
       "--audit-config", audit_config, "--output-dir", results)
print("Pilot artifacts saved in", results)
print("Stop here. No training was launched.")
